In [6]:
# ============================================================
# PHASE 5A — CELL 1: SETUP & DATA INGESTION (UPDATED)
# ============================================================
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = Path(r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml")
FEATURE_FILE = BASE_DIR / "data" / "raw" / "amazon_metadata_26_features.csv.gz"
MODEL_FILE = BASE_DIR / "models" / "trustguard_metadata_isolation_forest.joblib"
REPORT_DIR = BASE_DIR / "reports"
MODEL_DIR = BASE_DIR / "models"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not FEATURE_FILE.exists() or not MODEL_FILE.exists():
    raise FileNotFoundError("Missing feature dataset or model file in specified path.")

df = pd.read_csv(FEATURE_FILE, compression="gzip")

# Load artifact dictionary/model
loaded_artifact = joblib.load(MODEL_FILE)
saved_features = None

if isinstance(loaded_artifact, dict):
    model = loaded_artifact.get("model") or loaded_artifact.get("isolation_forest") or list(loaded_artifact.values())[0]
    saved_features = loaded_artifact.get("features") or loaded_artifact.get("feature_names") or loaded_artifact.get("feature_columns")
else:
    model = loaded_artifact

# Identify feature columns
if saved_features and len(saved_features) == getattr(model, "n_features_in_", 22):
    feature_columns = saved_features
else:
    NON_FEATURE_COLUMNS = [
        "parent_asin", "asin", "product_id", "product_ID", 
        "fake", "label", "risk_score", "anomaly_score", "risk_level", "is_anomaly"
    ]
    candidate_cols = [c for c in df.columns if c not in NON_FEATURE_COLUMNS and pd.api.types.is_numeric_dtype(df[c])]
    
    # Trim to match model's expected input dimension (22)
    expected_n = getattr(model, "n_features_in_", 22)
    feature_columns = candidate_cols[:expected_n]

X = df[feature_columns].copy()

print("=" * 80)
print("PHASE 5A LOADED SUCCESSFULLY")
print("=" * 80)
print(f"Dataset Shape    : {df.shape}")
print(f"Model Expected   : {getattr(model, 'n_features_in_', 'Unknown')} features")
print(f"Selected Features: {len(feature_columns)}")
print(f"Missing Values   : {X.isna().sum().sum()}")

PHASE 5A LOADED SUCCESSFULLY
Dataset Shape    : (10000, 27)
Model Expected   : 22 features
Selected Features: 22
Missing Values   : 22872


In [7]:
# ============================================================
# PHASE 5A — CELL 2: ISOLATION FOREST & DEVIATION MODEL (UPDATED)
# ============================================================
X_model = X.fillna(0)

# Pass raw NumPy array (.values) to avoid feature name/dimension warnings
X_array = X_model.values

validation_df = df.copy()
validation_df["if_prediction"] = model.predict(X_array)
validation_df["if_decision_score"] = model.decision_function(X_array)
validation_df["if_score_samples"] = model.score_samples(X_array)
validation_df["is_anomaly"] = (validation_df["if_prediction"] == -1).astype(int)

# Fit Robust Statistics (Median + MAD) for feature explanations
robust_stats = {}
for col in feature_columns:
    series = pd.to_numeric(X_model[col], errors="coerce")
    median = series.median()
    mad = np.median(np.abs(series.dropna() - median))
    if pd.isna(mad) or mad == 0:
        mad = 1.0
    robust_stats[col] = {"median": float(median), "mad": float(mad)}

def robust_deviation(row, feature):
    val = row[feature]
    if pd.isna(val): return 0.0
    return abs(val - robust_stats[feature]["median"]) / robust_stats[feature]["mad"]

def explain_product(row, top_n=5):
    deviations = []
    for feature in feature_columns:
        val = row[feature]
        if pd.isna(val): continue
        deviations.append({
            "feature": feature,
            "value": float(val),
            "deviation": float(robust_deviation(row, feature))
        })
    deviations.sort(key=lambda x: x["deviation"], reverse=True)
    return deviations[:top_n]

print("=" * 80)
print("MODEL INFERENCE COMPLETE")
print("=" * 80)
print(f"Normal Products  : {(validation_df['if_prediction'] == 1).sum()}")
print(f"Anomalies        : {(validation_df['if_prediction'] == -1).sum()}")

MODEL INFERENCE COMPLETE
Normal Products  : 1261
Anomalies        : 8739


In [8]:
# ============================================================
# PHASE 5A — CELL 3: RISK SCORE CALIBRATION & HEURISTICS
# ============================================================
# Calibrate anomaly score to 0-100
decision_values = validation_df["if_decision_score"]
low = decision_values.quantile(0.01)
high = decision_values.quantile(0.99)

def decision_to_anomaly_score(val):
    norm = 1 - ((val - low) / (high - low))
    return float(np.clip(norm, 0, 1) * 100)

validation_df["anomaly_score"] = validation_df["if_decision_score"].apply(decision_to_anomaly_score)

# Risk Indicators (40% component)
indicator_weights = {
    "extreme_rating": 1.0, "very_low_review_count": 1.0,
    "high_rating_low_reviews": 1.5, "missing_seller": 1.0,
    "missing_price": 0.5, "price_anomaly": 1.5,
    "very_short_title": 0.5, "very_short_description": 0.5,
}
total_weight = sum(indicator_weights.values())

def generate_risk_indicators(row):
    return {
        "extreme_rating": int(row.get("average_rating", 3) >= 4.8 or row.get("average_rating", 3) <= 1.5),
        "very_low_review_count": int(row.get("rating_number", 10) <= 5),
        "high_rating_low_reviews": int(row.get("average_rating", 0) >= 4.5 and row.get("rating_number", 99) <= 10),
        "missing_seller": int(row.get("seller_missing", 0) == 1),
        "missing_price": int(row.get("price_missing", 0) == 1),
        "price_anomaly": int(row.get("price_anomaly", 0) == 1),
        "very_short_title": int(row.get("title_length", 100) < 10),
        "very_short_description": int(row.get("description_length", 100) < 20)
    }

def calculate_indicator_score(row):
    inds = generate_risk_indicators(row)
    weighted_sum = sum(inds[k] * indicator_weights[k] for k in indicator_weights if k in inds)
    return (weighted_sum / total_weight) * 100

validation_df["indicator_score"] = validation_df.apply(calculate_indicator_score, axis=1)

# Combined TrustGuard Risk Score (60/40 Split)
validation_df["risk_score"] = (0.60 * validation_df["anomaly_score"] + 0.40 * validation_df["indicator_score"]).clip(0, 100)

def get_risk_level(score):
    if score < 25: return "LOW"
    elif score < 50: return "MEDIUM"
    elif score < 75: return "HIGH"
    else: return "CRITICAL"

validation_df["risk_level"] = validation_df["risk_score"].apply(get_risk_level)

print("=" * 80)
print("RISK SCORE CALIBRATION COMPLETE")
print("=" * 80)
display(validation_df[["risk_score", "anomaly_score", "indicator_score"]].describe())
print("\nRisk Level Distribution:")
print(validation_df["risk_level"].value_counts().to_dict())

RISK SCORE CALIBRATION COMPLETE


,risk_score,anomaly_score,indicator_score
count,10000.000000,10000.000000,10000.000000
mean,33.890459,46.349432,15.202000
std,12.602835,19.067690,15.898988
min,0.000000,0.000000,0.000000
25%,26.002987,34.634078,6.666667
50%,33.112660,45.433891,6.666667
75%,41.555047,57.291819,20.000000
max,89.333333,100.000000,73.333333



Risk Level Distribution:
{'MEDIUM': 6788, 'LOW': 2180, 'HIGH': 1004, 'CRITICAL': 28}


In [9]:
# ============================================================
# PHASE 5A — CELL 4: SYNTHETIC SANITY VALIDATION (UPDATED)
# ============================================================
normal_case = pd.Series({f: X_model[f].median() for f in feature_columns})

suspicious_case = normal_case.copy()
for key in ["average_rating", "rating_extremeness", "high_rating", "low_review_count", 
            "high_rating_low_reviews", "seller_missing", "price_anomaly", "price_missing"]:
    if key in suspicious_case.index:
        suspicious_case[key] = 1.0 if key != "average_rating" else 5.0
if "rating_number" in suspicious_case.index:
    suspicious_case["rating_number"] = 1.0

def evaluate_case(case_series):
    case_array = case_series.values.reshape(1, -1)
    decision = float(model.decision_function(case_array)[0])
    anomaly_sc = decision_to_anomaly_score(decision)
    indicator_sc = calculate_indicator_score(case_series)
    final_sc = float(np.clip(0.60 * anomaly_sc + 0.40 * indicator_sc, 0, 100))
    return decision, anomaly_sc, indicator_sc, final_sc, get_risk_level(final_sc)

norm_res = evaluate_case(normal_case)
susp_res = evaluate_case(suspicious_case)

sanity_df = pd.DataFrame([
    {"case": "NORMAL", "decision": norm_res[0], "anomaly_score": norm_res[1], "indicator_score": norm_res[2], "risk_score": norm_res[3], "risk_level": norm_res[4]},
    {"case": "SUSPICIOUS", "decision": susp_res[0], "anomaly_score": susp_res[1], "indicator_score": susp_res[2], "risk_score": susp_res[3], "risk_level": susp_res[4]}
])

print("=" * 80)
print("SYNTHETIC SANITY CHECK")
print("=" * 80)
display(sanity_df)

if susp_res[3] > norm_res[3]:
    print("\n✓ SANITY CHECK PASSED: Suspicious synthetic product correctly flagged higher.")
else:
    print("\n⚠ SANITY CHECK FAILED: Review model features.")

SYNTHETIC SANITY CHECK


,case,decision,anomaly_score,indicator_score,risk_score,risk_level
0,NORMAL,0.012756,15.855788,6.666667,12.180139,LOW
1,SUSPICIOUS,-0.015921,39.336387,86.666667,58.268499,HIGH



✓ SANITY CHECK PASSED: Suspicious synthetic product correctly flagged higher.


In [10]:
# ============================================================
# PHASE 5A — CELL 5: API CONTRACT & ARTIFACT EXPORT
# ============================================================
VALIDATION_FILE = BASE_DIR / "data" / "raw" / "phase5a_validation_results.csv.gz"
EXPLANATION_FILE = REPORT_DIR / "phase5a_top_risk_explanations.csv"
CALIBRATION_FILE = MODEL_DIR / "trustguard_risk_calibration.json"
FINAL_REPORT_FILE = REPORT_DIR / "phase5a_final_validation_report.json"

# Export validation records
validation_df.to_csv(VALIDATION_FILE, index=False, compression="gzip")

# Export top risk explanations
top_risk = validation_df.sort_values("risk_score", ascending=False).head(20)
exp_rows = []
for idx, row in top_risk.iterrows():
    p_id = row.get("parent_asin", idx)
    for rank, item in enumerate(explain_product(row, top_n=5), start=1):
        exp_rows.append({"product_id": p_id, "risk_score": row["risk_score"], "rank": rank, **item})
pd.DataFrame(exp_rows).to_csv(EXPLANATION_FILE, index=False)

# Export calibration configuration for FastAPI endpoint
calibration_config = {
    "version": "phase5a_v1",
    "anomaly_score_bounds": {"lower_quantile_01": float(low), "upper_quantile_99": float(high)},
    "risk_weights": {"isolation_forest": 0.60, "risk_indicators": 0.40},
    "risk_levels": {"LOW": [0, 25], "MEDIUM": [25, 50], "HIGH": [50, 75], "CRITICAL": [75, 100]},
    "indicator_weights": indicator_weights,
    "feature_columns": feature_columns
}
with open(CALIBRATION_FILE, "w", encoding="utf-8") as f:
    json.dump(calibration_config, f, indent=4)

# Final JSON summary report
final_report = {
    "phase": "5A",
    "dataset_rows": len(validation_df),
    "model_features": len(feature_columns),
    "anomalies_detected": int(validation_df["is_anomaly"].sum()),
    "synthetic_sanity_check_passed": bool(susp_res[3] > norm_res[3]),
    "artifacts_saved": [str(VALIDATION_FILE), str(EXPLANATION_FILE), str(CALIBRATION_FILE)]
}
with open(FINAL_REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=4)

print("=" * 80)
print("PHASE 5A COMPLETE — ALL ARTIFACTS SAVED")
print("=" * 80)
print(json.dumps(final_report, indent=4))

PHASE 5A COMPLETE — ALL ARTIFACTS SAVED
{
    "phase": "5A",
    "dataset_rows": 10000,
    "model_features": 22,
    "anomalies_detected": 8739,
    "synthetic_sanity_check_passed": true,
    "artifacts_saved": [
        "C:\\Users\\Aman\\Desktop\\TrustGuard\\product_scam_ml\\data\\raw\\phase5a_validation_results.csv.gz",
        "C:\\Users\\Aman\\Desktop\\TrustGuard\\product_scam_ml\\reports\\phase5a_top_risk_explanations.csv",
        "C:\\Users\\Aman\\Desktop\\TrustGuard\\product_scam_ml\\models\\trustguard_risk_calibration.json"
    ]
}


In [12]:
# ============================================================
# PHASE 5A — CELL 1 (CORRECTED)
# INSPECT SAVED MODEL ARTIFACT
# ============================================================

import joblib
import numpy as np
import pandas as pd

MODEL_PATH = r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\models\trustguard_metadata_isolation_forest.joblib"

print("=" * 80)
print("PHASE 5A — SAVED MODEL ARTIFACT INSPECTION")
print("=" * 80)

artifact = joblib.load(MODEL_PATH)

print("\nArtifact type:")
print(type(artifact))

# ------------------------------------------------------------
# CASE 1 — Dictionary artifact
# ------------------------------------------------------------

if isinstance(artifact, dict):

    print("\nArtifact keys:")
    for key in artifact.keys():
        value = artifact[key]

        if isinstance(value, (list, tuple, np.ndarray)):
            try:
                shape = np.asarray(value).shape
            except Exception:
                shape = "N/A"

            print(f"  {key:<30} {type(value).__name__} shape={shape}")

        else:
            print(f"  {key:<30} {type(value).__name__}")

    # Try to locate the actual Isolation Forest
    model_candidates = []

    for key, value in artifact.items():
        if hasattr(value, "predict") and hasattr(value, "decision_function"):
            model_candidates.append((key, value))

    print("\nPotential fitted model objects:")

    if not model_candidates:
        print("  NONE FOUND")

    else:
        for key, value in model_candidates:
            print(f"  {key} -> {type(value)}")

    # Select the Isolation Forest
    if "model" in artifact and hasattr(artifact["model"], "predict"):
        model = artifact["model"]

    elif model_candidates:
        model = model_candidates[0][1]

    else:
        raise ValueError(
            "Could not find the fitted IsolationForest inside the saved artifact."
        )

# ------------------------------------------------------------
# CASE 2 — Raw model artifact
# ------------------------------------------------------------

else:

    model = artifact

print("\n" + "=" * 80)
print("FITTED MODEL CONFIGURATION")
print("=" * 80)

print("Model object      :", type(model))

print("n_estimators      :", getattr(model, "n_estimators", "N/A"))
print("contamination     :", getattr(model, "contamination", "N/A"))
print("max_samples       :", getattr(model, "max_samples", "N/A"))
print("random_state      :", getattr(model, "random_state", "N/A"))

print("\nModel feature count:")

if hasattr(model, "n_features_in_"):
    print("n_features_in_    :", model.n_features_in_)
else:
    print("n_features_in_    : NOT AVAILABLE")

print("\nModel fitted:")
print(hasattr(model, "estimators_"))

PHASE 5A — SAVED MODEL ARTIFACT INSPECTION

Artifact type:
<class 'dict'>

Artifact keys:
  model                          IsolationForest
  imputer                        SimpleImputer
  feature_columns                list shape=(26,)
  model_features                 list shape=(22,)
  constant_features              list shape=(0,)
  high_missing_features          list shape=(4,)
  global_price_median            float
  category_price_median          dict
  risk_weights                   dict
  anomaly_weight                 float
  rule_weight                    float
  anomaly_reference              list shape=(10000,)
  model_type                     str
  n_estimators                   int
  contamination                  float
  random_state                   int
  dataset                        str
  training_rows                  int
  architecture                   str

Potential fitted model objects:
  model -> <class 'sklearn.ensemble._iforest.IsolationForest'>

FITTED MODEL

In [13]:
# ============================================================
# PHASE 5A — CELL 2
# FEATURE ORDER / SCHEMA VALIDATION
# ============================================================

print("=" * 80)
print("FEATURE SCHEMA VALIDATION")
print("=" * 80)

# Load the feature dataset used during Phase 4B.2
FEATURE_PATH = r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\data\raw\amazon_metadata_26_features.csv.gz"

df = pd.read_csv(FEATURE_PATH)

stored_all_features = artifact["feature_columns"]
stored_model_features = artifact["model_features"]
stored_constant_features = artifact["constant_features"]
stored_high_missing = artifact["high_missing_features"]

print("\nDataset shape:")
print(df.shape)

print("\nStored feature columns:")
print(len(stored_all_features))
print(stored_all_features)

print("\nStored model features:")
print(len(stored_model_features))
for i, col in enumerate(stored_model_features):
    print(f"{i:02d} -> {col}")

print("\nCurrent dataset columns:")
print(len(df.columns))

# ------------------------------------------------------------
# CHECK 1 — Are all stored model features present?
# ------------------------------------------------------------

missing_from_dataset = [
    col for col in stored_model_features
    if col not in df.columns
]

extra_columns = [
    col for col in df.columns
    if col not in stored_model_features
]

print("\n" + "-" * 80)
print("FEATURE PRESENCE")
print("-" * 80)

print("Missing model features:")
print(missing_from_dataset)

print("\nDataset columns not used by model:")
print(extra_columns)

# ------------------------------------------------------------
# CHECK 2 — Exact feature ordering
# ------------------------------------------------------------

dataset_model_columns = [
    col for col in df.columns
    if col in stored_model_features
]

print("\n" + "-" * 80)
print("FEATURE ORDER COMPARISON")
print("-" * 80)

print("Stored order:")
print(stored_model_features)

print("\nDataset-derived order:")
print(dataset_model_columns)

print("\nExact order match:")
print(stored_model_features == dataset_model_columns)

# ------------------------------------------------------------
# CHECK 3 — Build X using STORED order
# ------------------------------------------------------------

X_raw = df[stored_model_features].copy()

print("\n" + "-" * 80)
print("MODEL INPUT")
print("-" * 80)

print("X_raw shape:")
print(X_raw.shape)

print("\nColumns passed to model:")
for i, col in enumerate(X_raw.columns):
    print(f"{i:02d} -> {col}")

# ------------------------------------------------------------
# CHECK 4 — Missing values
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("MISSING VALUES BEFORE IMPUTATION")
print("-" * 80)

missing_table = (
    X_raw.isna()
    .sum()
    .to_frame("missing_count")
)

missing_table["missing_pct"] = (
    missing_table["missing_count"] / len(X_raw) * 100
)

display(missing_table)

# ------------------------------------------------------------
# CHECK 5 — Compare against saved imputer
# ------------------------------------------------------------

imputer = artifact["imputer"]

print("\n" + "-" * 80)
print("IMPUTER")
print("-" * 80)

print("Imputer type:")
print(type(imputer))

print("Imputer feature count:")
print(getattr(imputer, "n_features_in_", "N/A"))

if hasattr(imputer, "feature_names_in_"):
    print("\nImputer feature order:")
    for i, col in enumerate(imputer.feature_names_in_):
        print(f"{i:02d} -> {col}")

# ------------------------------------------------------------
# CHECK 6 — Transform using the SAVED imputer
# ------------------------------------------------------------

X_imputed = imputer.transform(X_raw)

print("\nImputed matrix shape:")
print(X_imputed.shape)

print("\nExpected model feature count:")
print(model.n_features_in_)

print("\nShape compatible with model:")
print(X_imputed.shape[1] == model.n_features_in_)

FEATURE SCHEMA VALIDATION

Dataset shape:
(10000, 27)

Stored feature columns:
26
['title_length', 'title_word_count', 'uppercase_ratio', 'special_character_ratio', 'description_length', 'description_word_count', 'feature_count', 'feature_text_length', 'category_count', 'image_count', 'video_count', 'has_videos', 'seller_missing', 'seller_name_length', 'price_numeric', 'price_missing', 'price_ratio_to_category', 'log_price_ratio', 'price_anomaly', 'average_rating', 'rating_number', 'log_rating_number', 'rating_extremeness', 'high_rating', 'low_review_count', 'high_rating_low_reviews']

Stored model features:
22
00 -> title_length
01 -> title_word_count
02 -> uppercase_ratio
03 -> special_character_ratio
04 -> description_length
05 -> description_word_count
06 -> feature_count
07 -> feature_text_length
08 -> category_count
09 -> image_count
10 -> video_count
11 -> has_videos
12 -> seller_missing
13 -> seller_name_length
14 -> price_missing
15 -> average_rating
16 -> rating_number
17 -> 

,missing_count,missing_pct
title_length,0,0.0
title_word_count,0,0.0
uppercase_ratio,0,0.0
special_character_ratio,0,0.0
description_length,0,0.0
description_word_count,0,0.0
feature_count,0,0.0
feature_text_length,0,0.0
category_count,0,0.0
image_count,0,0.0



--------------------------------------------------------------------------------
IMPUTER
--------------------------------------------------------------------------------
Imputer type:
<class 'sklearn.impute._base.SimpleImputer'>
Imputer feature count:
22

Imputer feature order:
00 -> title_length
01 -> title_word_count
02 -> uppercase_ratio
03 -> special_character_ratio
04 -> description_length
05 -> description_word_count
06 -> feature_count
07 -> feature_text_length
08 -> category_count
09 -> image_count
10 -> video_count
11 -> has_videos
12 -> seller_missing
13 -> seller_name_length
14 -> price_missing
15 -> average_rating
16 -> rating_number
17 -> log_rating_number
18 -> rating_extremeness
19 -> high_rating
20 -> low_review_count
21 -> high_rating_low_reviews

Imputed matrix shape:
(10000, 22)

Expected model feature count:
22

Shape compatible with model:
True


In [14]:
# ============================================================
# PHASE 5A.1 — DIRECT MODEL PREDICTION AUDIT
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

MODEL_PATH = r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\models\trustguard_metadata_isolation_forest.joblib"

FEATURE_DATA_PATH = (
    r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\data\raw"
    r"\amazon_metadata_26_features.csv.gz"
)

artifact = joblib.load(MODEL_PATH)
df = pd.read_csv(FEATURE_DATA_PATH, compression="gzip")

model = artifact["model"]
imputer = artifact["imputer"]
model_features = artifact["model_features"]

print("=" * 80)
print("DIRECT MODEL PREDICTION AUDIT")
print("=" * 80)

print("Rows:", len(df))
print("Model features:", len(model_features))
print("Model type:", type(model))
print("Contamination:", model.contamination)

X_raw = df[model_features].copy()

X_imputed = imputer.transform(X_raw)

print("X_raw shape:", X_raw.shape)
print("X_imputed shape:", X_imputed.shape)

# Direct sklearn prediction
direct_prediction = model.predict(X_imputed)

print("\nPrediction distribution:")
print(pd.Series(direct_prediction).value_counts().sort_index())

anomaly_count = np.sum(direct_prediction == -1)
normal_count = np.sum(direct_prediction == 1)

print("\nDirect sklearn results:")
print("Normal (+1):", normal_count)
print("Anomaly (-1):", anomaly_count)
print("Anomaly %:", anomaly_count / len(df) * 100)

DIRECT MODEL PREDICTION AUDIT
Rows: 10000
Model features: 22
Model type: <class 'sklearn.ensemble._iforest.IsolationForest'>
Contamination: 0.05
X_raw shape: (10000, 22)
X_imputed shape: (10000, 22)

Prediction distribution:
-1     500
 1    9500
Name: count, dtype: int64

Direct sklearn results:
Normal (+1): 9500
Anomaly (-1): 500
Anomaly %: 5.0


In [15]:
# ============================================================
# ISOLATION FOREST SCORE AUDIT
# ============================================================

print("=" * 80)
print("ISOLATION FOREST SCORE AUDIT")
print("=" * 80)

score_samples = model.score_samples(X_imputed)
decision_scores = model.decision_function(X_imputed)

print("score_samples:")
print(pd.Series(score_samples).describe())

print("\ndecision_function:")
print(pd.Series(decision_scores).describe())

print("\nModel offset:")
print(model.offset_)

print("\nScore relationship:")
print(
    "decision_function = score_samples - offset"
)

print(
    "\nCheck:",
    np.allclose(
        decision_scores,
        score_samples - model.offset_
    )
)

print("\nManual prediction from decision_function:")

manual_prediction = np.where(
    decision_scores >= 0,
    1,
    -1
)

print(pd.Series(manual_prediction).value_counts().sort_index())

print(
    "\nManual anomaly %:",
    np.mean(manual_prediction == -1) * 100
)

ISOLATION FOREST SCORE AUDIT
score_samples:
count    10000.000000
mean        -0.462811
std          0.036634
min         -0.648764
25%         -0.485790
50%         -0.458532
75%         -0.434884
max         -0.389913
dtype: float64

decision_function:
count    10000.000000
mean         0.067788
std          0.036634
min         -0.118164
25%          0.044810
50%          0.072068
75%          0.095716
max          0.140687
dtype: float64

Model offset:
-0.5305998787686055

Score relationship:
decision_function = score_samples - offset

Check: True

Manual prediction from decision_function:
-1     500
 1    9500
Name: count, dtype: int64

Manual anomaly %: 5.0


In [16]:
# ============================================================
# PREDICTION CONSISTENCY CHECK
# ============================================================

print("=" * 80)
print("PREDICTION CONSISTENCY CHECK")
print("=" * 80)

sklearn_prediction = model.predict(X_imputed)

difference_count = np.sum(
    sklearn_prediction != manual_prediction
)

print("Sklearn prediction count:")
print(pd.Series(sklearn_prediction).value_counts().sort_index())

print("\nManual prediction count:")
print(pd.Series(manual_prediction).value_counts().sort_index())

print("\nPrediction disagreements:", difference_count)

if difference_count == 0:
    print("\n✓ Manual threshold exactly matches sklearn predict().")
else:
    print("\n⚠ Prediction mismatch detected.")

PREDICTION CONSISTENCY CHECK
Sklearn prediction count:
-1     500
 1    9500
Name: count, dtype: int64

Manual prediction count:
-1     500
 1    9500
Name: count, dtype: int64

Prediction disagreements: 0

✓ Manual threshold exactly matches sklearn predict().


In [17]:
# ============================================================
# INSPECT PHASE 5A RESULTS
# ============================================================

VALIDATION_PATH = (
    r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\data\raw"
    r"\phase5a_validation_results.csv.gz"
)

validation = pd.read_csv(
    VALIDATION_PATH,
    compression="gzip"
)

print("=" * 80)
print("PHASE 5A RESULT INSPECTION")
print("=" * 80)

print("Shape:", validation.shape)

print("\nColumns:")
for i, col in enumerate(validation.columns):
    print(f"{i:02d} -> {col}")

print("\nFirst 5 rows:")
display(validation.head())

PHASE 5A RESULT INSPECTION
Shape: (10000, 35)

Columns:
00 -> parent_asin
01 -> title_length
02 -> title_word_count
03 -> uppercase_ratio
04 -> special_character_ratio
05 -> description_length
06 -> description_word_count
07 -> feature_count
08 -> feature_text_length
09 -> category_count
10 -> image_count
11 -> video_count
12 -> has_videos
13 -> seller_missing
14 -> seller_name_length
15 -> price_numeric
16 -> price_missing
17 -> price_ratio_to_category
18 -> log_price_ratio
19 -> price_anomaly
20 -> average_rating
21 -> rating_number
22 -> log_rating_number
23 -> rating_extremeness
24 -> high_rating
25 -> low_review_count
26 -> high_rating_low_reviews
27 -> if_prediction
28 -> if_decision_score
29 -> if_score_samples
30 -> is_anomaly
31 -> anomaly_score
32 -> indicator_score
33 -> risk_score
34 -> risk_level

First 5 rows:


,parent_asin,title_length,title_word_count,uppercase_ratio,special_character_ratio,description_length,description_word_count,feature_count,feature_text_length,category_count,...,low_review_count,high_rating_low_reviews,if_prediction,if_decision_score,if_score_samples,is_anomaly,anomaly_score,indicator_score,risk_score,risk_level
0,B00MCW7G9M,38,6,1.000000,0.026316,658,105,0,0,3,...,1,0,-1,-0.067309,-0.597909,1,81.412438,6.666667,51.514129,HIGH
1,B00YT6XQSE,29,6,0.500000,0.068966,18,4,2,35,5,...,1,1,-1,-0.040637,-0.571236,1,59.573273,60.000000,59.743964,HIGH
2,B07SM135LS,202,31,0.194631,0.049505,0,0,5,823,5,...,0,0,-1,-0.010940,-0.541540,1,35.258366,6.666667,23.821686,LOW
3,B089CNGZCW,199,28,0.142857,0.015075,0,0,5,1102,3,...,0,0,1,0.004554,-0.526046,0,22.572145,6.666667,16.209953,LOW
4,B004E2Z88O,38,6,0.181818,0.000000,242,34,3,142,5,...,0,0,1,0.015243,-0.515357,0,13.820208,0.000000,8.292125,LOW


In [18]:
# ============================================================
# TRACE ANOMALY COUNT
# ============================================================

print("=" * 80)
print("TRACEING THE 87.4% ANOMALY COUNT")
print("=" * 80)

for col in validation.columns:

    try:

        values = validation[col]

        unique_values = values.dropna().unique()

        # Look specifically for -1 / +1 style predictions
        if set(unique_values).issubset({-1, 1}):

            anomaly_count = np.sum(values == -1)

            print(
                f"\nColumn: {col}"
            )

            print(
                "Unique:",
                sorted(unique_values)
            )

            print(
                "Anomaly (-1):",
                anomaly_count
            )

            print(
                "Anomaly %:",
                anomaly_count / len(validation) * 100
            )

    except Exception:
        pass

TRACEING THE 87.4% ANOMALY COUNT

Column: if_prediction
Unique: [np.int64(-1), np.int64(1)]
Anomaly (-1): 8739
Anomaly %: 87.39


In [19]:
# ============================================================
# SAVED VS FRESH PREDICTION COMPARISON
# ============================================================

print("=" * 80)
print("SAVED VS FRESH MODEL PREDICTIONS")
print("=" * 80)

fresh_prediction = model.predict(X_imputed)

if "if_prediction" in validation.columns:

    saved_prediction = validation["if_prediction"].values

    print("Saved prediction distribution:")
    print(
        pd.Series(saved_prediction)
        .value_counts()
        .sort_index()
    )

    print("\nFresh model prediction distribution:")
    print(
        pd.Series(fresh_prediction)
        .value_counts()
        .sort_index()
    )

    # Compare row-by-row
    comparison_count = min(
        len(saved_prediction),
        len(fresh_prediction)
    )

    disagreements = np.sum(
        saved_prediction[:comparison_count]
        != fresh_prediction[:comparison_count]
    )

    print(
        "\nRow-by-row disagreements:",
        disagreements,
        "/",
        comparison_count
    )

    print(
        "Agreement:",
        (1 - disagreements / comparison_count) * 100,
        "%"
    )

else:

    print(
        "Column 'if_prediction' does not exist "
        "in Phase 5A validation results."
    )

SAVED VS FRESH MODEL PREDICTIONS
Saved prediction distribution:
-1    8739
 1    1261
Name: count, dtype: int64

Fresh model prediction distribution:
-1     500
 1    9500
Name: count, dtype: int64

Row-by-row disagreements: 8249 / 10000
Agreement: 17.510000000000005 %


In [20]:
# ============================================================
# CONTAMINATION / OFFSET DIAGNOSTIC
# ============================================================

print("=" * 80)
print("CONTAMINATION / OFFSET DIAGNOSTIC")
print("=" * 80)

print("Configured contamination:")
print(model.contamination)

print("\nLearned offset:")
print(model.offset_)

scores = model.score_samples(X_imputed)

below_offset = np.sum(
    scores < model.offset_
)

above_or_equal_offset = np.sum(
    scores >= model.offset_
)

print("\nScores below offset:")
print(below_offset)

print("Scores >= offset:")
print(above_or_equal_offset)

print(
    "\nPercentage below offset:",
    below_offset / len(scores) * 100
)

CONTAMINATION / OFFSET DIAGNOSTIC
Configured contamination:
0.05

Learned offset:
-0.5305998787686055

Scores below offset:
500
Scores >= offset:
9500

Percentage below offset: 5.0


In [21]:
# ============================================================
# ANOMALY REFERENCE AUDIT
# ============================================================

print("=" * 80)
print("ANOMALY REFERENCE AUDIT")
print("=" * 80)

reference = np.asarray(
    artifact["anomaly_reference"]
)

print("Reference shape:", reference.shape)
print("Reference dtype:", reference.dtype)

print("\nReference statistics:")
print(
    pd.Series(reference).describe()
)

print("\nReference unique count:")
print(
    len(np.unique(reference))
)

print("\nReference first 20 values:")
print(reference[:20])

ANOMALY REFERENCE AUDIT
Reference shape: (10000,)
Reference dtype: float64

Reference statistics:
count    10000.000000
mean        -0.067788
std          0.036634
min         -0.140687
25%         -0.095716
50%         -0.072068
75%         -0.044810
max          0.118164
dtype: float64

Reference unique count:
9996

Reference first 20 values:
[-0.02870167 -0.02614881 -0.10648671 -0.07128201 -0.09297862 -0.11214915
 -0.0918939  -0.08595435 -0.07235046 -0.09813121 -0.03788962 -0.12045292
 -0.04588524 -0.12021883 -0.08528862 -0.09537367 -0.00097493 -0.02997233
 -0.09776328 -0.10630649]


In [22]:
# ============================================================
# CLASSIFY ANOMALY REFERENCE
# ============================================================

print("=" * 80)
print("ANOMALY REFERENCE TYPE")
print("=" * 80)

unique_reference = np.unique(reference)

if set(unique_reference).issubset({-1, 1}):

    print("Reference appears to contain +/-1 predictions.")

    print(
        pd.Series(reference)
        .value_counts()
        .sort_index()
    )

elif np.issubdtype(reference.dtype, np.number):

    print("Reference appears to contain continuous scores.")

    print(
        pd.Series(reference).describe()
    )

else:

    print("Reference contains non-numeric values.")

ANOMALY REFERENCE TYPE
Reference appears to contain continuous scores.
count    10000.000000
mean        -0.067788
std          0.036634
min         -0.140687
25%         -0.095716
50%         -0.072068
75%         -0.044810
max          0.118164
dtype: float64


In [23]:
# ============================================================
# PHASE 5A FIX — AUTHORITATIVE ISOLATION FOREST PREDICTION
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix

BASE_DIR = r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml"

MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "trustguard_metadata_isolation_forest.joblib"
)

FEATURE_DATASET = os.path.join(
    BASE_DIR,
    "data",
    "raw",
    "amazon_metadata_26_features.csv.gz"
)

REPORT_DIR = os.path.join(BASE_DIR, "reports")
DATA_DIR = os.path.join(BASE_DIR, "data", "raw")

os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


# ============================================================
# 1. LOAD ARTIFACT
# ============================================================

artifact = joblib.load(MODEL_PATH)

model = artifact["model"]
imputer = artifact["imputer"]

feature_columns = artifact["feature_columns"]
model_features = artifact["model_features"]

print("=" * 80)
print("PHASE 5A FIX — AUTHORITATIVE MODEL PREDICTION")
print("=" * 80)

print("Model:", type(model))
print("Contamination:", model.contamination)
print("Model features:", len(model_features))
print("Feature order:", model_features)


# ============================================================
# 2. LOAD FEATURE DATASET
# ============================================================

df = pd.read_csv(FEATURE_DATASET)

print("\nDataset shape:", df.shape)


# ============================================================
# 3. VALIDATE FEATURE SCHEMA
# ============================================================

missing_features = [
    col for col in model_features
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing model features: {missing_features}"
    )

X_raw = df[model_features].copy()

print("\nX_raw shape:", X_raw.shape)


# ============================================================
# 4. APPLY THE ORIGINAL IMPUTER
# ============================================================

X_imputed = imputer.transform(X_raw)

print("X_imputed shape:", X_imputed.shape)

if X_imputed.shape[1] != model.n_features_in_:
    raise ValueError(
        f"Feature count mismatch: "
        f"{X_imputed.shape[1]} vs "
        f"{model.n_features_in_}"
    )


# ============================================================
# 5. AUTHORITATIVE SKLEARN PREDICTION
# ============================================================

if_prediction = model.predict(X_imputed)

if_decision_score = model.decision_function(X_imputed)

if_score_samples = model.score_samples(X_imputed)

is_anomaly = (if_prediction == -1).astype(int)


# ============================================================
# 6. VERIFY SKLEARN RELATIONSHIP
# ============================================================

manual_prediction = np.where(
    if_decision_score < 0,
    -1,
    1
)

prediction_disagreement = np.sum(
    if_prediction != manual_prediction
)

print("\n" + "=" * 80)
print("AUTHORITATIVE PREDICTION")
print("=" * 80)

print(
    pd.Series(if_prediction)
    .value_counts()
    .sort_index()
)

print(
    "\nAnomalies:",
    np.sum(is_anomaly)
)

print(
    "Anomaly percentage:",
    np.mean(is_anomaly) * 100
)

print(
    "\nPrediction disagreements:",
    prediction_disagreement
)

assert prediction_disagreement == 0


# ============================================================
# 7. BUILD CLEAN RESULTS
# ============================================================

results = df.copy()

results["if_prediction"] = if_prediction
results["if_decision_score"] = if_decision_score
results["if_score_samples"] = if_score_samples
results["is_anomaly"] = is_anomaly


# ============================================================
# 8. ANOMALY SCORE
#
# Larger value = more anomalous.
#
# We use the decision score:
#
# negative -> anomaly
# positive -> normal
#
# Convert it to a bounded 0-100 score.
# ============================================================

decision_min = if_decision_score.min()
decision_max = if_decision_score.max()

if decision_max == decision_min:
    anomaly_score = np.zeros(len(results))

else:
    anomaly_score = (
        (decision_max - if_decision_score)
        /
        (decision_max - decision_min)
        * 100
    )

results["anomaly_score"] = np.clip(
    anomaly_score,
    0,
    100
)


# ============================================================
# 9. SAVE INTERMEDIATE VALIDATION RESULTS
# ============================================================

validation_path = os.path.join(
    DATA_DIR,
    "phase5a_validation_results_corrected.csv.gz"
)

results.to_csv(
    validation_path,
    index=False,
    compression="gzip"
)

print("\nSaved corrected validation:")
print(validation_path)


# ============================================================
# 10. FINAL SANITY CHECK
# ============================================================

print("\n" + "=" * 80)
print("FINAL SANITY CHECK")
print("=" * 80)

print(
    results["if_prediction"]
    .value_counts()
    .sort_index()
)

print(
    "\nExpected anomaly percentage:",
    model.contamination * 100
)

print(
    "Actual anomaly percentage:",
    results["is_anomaly"].mean() * 100
)

assert abs(
    results["is_anomaly"].mean()
    - model.contamination
) < 0.001

print(
    "\n✓ Isolation Forest anomaly rate is consistent "
    "with configured contamination."
)

print(
    "✓ Feature ordering validated."
)

print(
    "✓ Imputer validated."
)

print(
    "✓ Direct sklearn prediction used."
)

print(
    "✓ No anomaly_reference used for classification."
)

PHASE 5A FIX — AUTHORITATIVE MODEL PREDICTION
Model: <class 'sklearn.ensemble._iforest.IsolationForest'>
Contamination: 0.05
Model features: 22
Feature order: ['title_length', 'title_word_count', 'uppercase_ratio', 'special_character_ratio', 'description_length', 'description_word_count', 'feature_count', 'feature_text_length', 'category_count', 'image_count', 'video_count', 'has_videos', 'seller_missing', 'seller_name_length', 'price_missing', 'average_rating', 'rating_number', 'log_rating_number', 'rating_extremeness', 'high_rating', 'low_review_count', 'high_rating_low_reviews']

Dataset shape: (10000, 27)

X_raw shape: (10000, 22)
X_imputed shape: (10000, 22)

AUTHORITATIVE PREDICTION
-1     500
 1    9500
Name: count, dtype: int64

Anomalies: 500
Anomaly percentage: 5.0

Prediction disagreements: 0

Saved corrected validation:
C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\data\raw\phase5a_validation_results_corrected.csv.gz

FINAL SANITY CHECK
if_prediction
-1     500
 1    950

In [24]:
# ============================================================
# PHASE 5A.1 — RISK SCORE RECALIBRATION & SANITY VERIFICATION
# ============================================================
import os
import json
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml"
INPUT_FILE = os.path.join(BASE_DIR, "data", "raw", "phase5a_validation_results_corrected.csv.gz")
CALIBRATION_FILE = os.path.join(BASE_DIR, "models", "trustguard_risk_calibration.json")
REPORT_FILE = os.path.join(BASE_DIR, "reports", "phase5a1_calibration_report.json")

df = pd.read_csv(INPUT_FILE)

# 1. Calibrate Anomaly Score (0–100 Bounded)
decision_vals = df["if_decision_score"]
low_q = decision_vals.quantile(0.01)
high_q = decision_vals.quantile(0.99)

def scale_anomaly_score(score):
    # Quantile normalization: lower decision score = higher anomaly risk
    norm = 1.0 - ((score - low_q) / (high_q - low_q))
    return float(np.clip(norm, 0, 1) * 100)

df["anomaly_score"] = df["if_decision_score"].apply(scale_anomaly_score)

# 2. Heuristic Indicator Weights (40% Component)
indicator_weights = {
    "extreme_rating": 1.0, 
    "very_low_review_count": 1.0,
    "high_rating_low_reviews": 1.5, 
    "missing_seller": 1.0,
    "missing_price": 0.5, 
    "price_anomaly": 1.5,
    "very_short_title": 0.5, 
    "very_short_description": 0.5,
}
total_weight = sum(indicator_weights.values())

def compute_indicator_score(row):
    inds = {
        "extreme_rating": int(row.get("average_rating", 3) >= 4.8 or row.get("average_rating", 3) <= 1.5),
        "very_low_review_count": int(row.get("rating_number", 10) <= 5),
        "high_rating_low_reviews": int(row.get("average_rating", 0) >= 4.5 and row.get("rating_number", 99) <= 10),
        "missing_seller": int(row.get("seller_missing", 0) == 1),
        "missing_price": int(row.get("price_missing", 0) == 1),
        "price_anomaly": int(row.get("price_anomaly", 0) == 1),
        "very_short_title": int(row.get("title_length", 100) < 10),
        "very_short_description": int(row.get("description_length", 100) < 20)
    }
    weighted_sum = sum(inds[k] * indicator_weights[k] for k in indicator_weights if k in inds)
    return (weighted_sum / total_weight) * 100

df["indicator_score"] = df.apply(compute_indicator_score, axis=1)

# 3. Combine Scores (60/40 Split)
df["risk_score"] = (0.60 * df["anomaly_score"] + 0.40 * df["indicator_score"]).clip(0, 100)

def assign_risk_level(score):
    if score < 25: return "LOW"
    elif score < 50: return "MEDIUM"
    elif score < 75: return "HIGH"
    else: return "CRITICAL"

df["risk_level"] = df["risk_score"].apply(assign_risk_level)

# 4. Save Calibration Artifact for FastAPI
calibration_config = {
    "version": "phase5a1_v1",
    "bounds": {"low_q01": float(low_q), "high_q99": float(high_q)},
    "risk_weights": {"isolation_forest": 0.60, "risk_indicators": 0.40},
    "risk_thresholds": {"LOW": [0, 25], "MEDIUM": [25, 50], "HIGH": [50, 75], "CRITICAL": [75, 100]},
    "indicator_weights": indicator_weights
}

with open(CALIBRATION_FILE, "w", encoding="utf-8") as f:
    json.dump(calibration_config, f, indent=4)

print("=" * 80)
print("PHASE 5A.1 RECALIBRATION COMPLETE")
print("=" * 80)
print("Risk Level Breakdown:")
print(df["risk_level"].value_counts().to_string())
print("\nRisk Score Summary Statistics:")
display(df[["anomaly_score", "indicator_score", "risk_score"]].describe())
print(f"\n✓ Saved calibration config: {CALIBRATION_FILE}")

PHASE 5A.1 RECALIBRATION COMPLETE
Risk Level Breakdown:
risk_level
LOW         4946
MEDIUM      3629
HIGH        1351
CRITICAL      74

Risk Score Summary Statistics:


,anomaly_score,indicator_score,risk_score
count,10000.000000,10000.000000,10000.000000
mean,38.048042,15.202000,28.909625
std,22.359937,15.898988,16.905948
min,0.000000,0.000000,0.947337
25%,20.811166,6.666667,15.773913
50%,35.468230,6.666667,25.212569
75%,52.363538,20.000000,40.108820
max,100.000000,73.333333,89.333333



✓ Saved calibration config: C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\models\trustguard_risk_calibration.json


In [25]:
# ============================================================
# PHASE 5A.2 — RISK ENGINE AUDIT & SYNTHETIC VALIDATION
# ============================================================
import os
import json
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml"
INPUT_FILE = os.path.join(BASE_DIR, "data", "raw", "phase5a_validation_results_corrected.csv.gz")
CALIBRATION_FILE = os.path.join(BASE_DIR, "models", "trustguard_risk_calibration.json")
EXPLANATION_FILE = os.path.join(BASE_DIR, "reports", "phase5a2_synthetic_validation.csv")

df = pd.read_csv(INPUT_FILE)

# 1. Refined Indicator Weights (Price indicators excluded due to dataset sparsity)
indicator_weights = {
    "extreme_rating": 1.0,
    "very_low_review_count": 1.0,
    "high_rating_low_reviews": 1.5,
    "missing_seller": 1.0,
    "very_short_title": 0.5,
    "very_short_description": 0.5,
}
total_weight = sum(indicator_weights.values())

def extract_indicators(row):
    return {
        "extreme_rating": int(row.get("average_rating", 3) >= 4.8 or row.get("average_rating", 3) <= 1.5),
        "very_low_review_count": int(row.get("rating_number", 10) <= 5),
        "high_rating_low_reviews": int(row.get("average_rating", 0) >= 4.5 and row.get("rating_number", 99) <= 10),
        "missing_seller": int(row.get("seller_missing", 0) == 1),
        "very_short_title": int(row.get("title_length", 100) < 10),
        "very_short_description": int(row.get("description_length", 100) < 20)
    }

# Calculate indicator triggers across dataset
indicator_matrix = pd.DataFrame([extract_indicators(row) for _, row in df.iterrows()])
prevalence = (indicator_matrix.mean() * 100).round(2)

print("=" * 80)
print("PHASE 5A.2 — INDICATOR PREVALENCE AUDIT (% of 10,000 Products Triggered)")
print("=" * 80)
for k, v in prevalence.items():
    print(f"{k:<25}: {v:>6.2f}%")

# Recalculate Refined Scores
decision_vals = df["if_decision_score"]
low_q = decision_vals.quantile(0.01)
high_q = decision_vals.quantile(0.99)

def scale_anomaly_score(score):
    norm = 1.0 - ((score - low_q) / (high_q - low_q))
    return float(np.clip(norm, 0, 1) * 100)

df["anomaly_score"] = df["if_decision_score"].apply(scale_anomaly_score)

def compute_indicator_score(row):
    inds = extract_indicators(row)
    weighted_sum = sum(inds[k] * indicator_weights[k] for k in indicator_weights if k in inds)
    return (weighted_sum / total_weight) * 100

df["indicator_score"] = df.apply(compute_indicator_score, axis=1)
df["risk_score"] = (0.60 * df["anomaly_score"] + 0.40 * df["indicator_score"]).clip(0, 100)

def assign_risk_level(score):
    if score < 25: return "LOW"
    elif score < 50: return "MEDIUM"
    elif score < 75: return "HIGH"
    else: return "CRITICAL"

df["risk_level"] = df["risk_score"].apply(assign_risk_level)

# 2. Synthetic Profile Monotonicity Validation
synthetic_normal = {
    "average_rating": 4.2, "rating_number": 150, "seller_missing": 0, 
    "title_length": 45, "description_length": 120, "if_decision_score": 0.08
}
synthetic_moderate = {
    "average_rating": 4.9, "rating_number": 4, "seller_missing": 1, 
    "title_length": 25, "description_length": 50, "if_decision_score": 0.01
}
synthetic_critical = {
    "average_rating": 5.0, "rating_number": 1, "seller_missing": 1, 
    "title_length": 8, "description_length": 12, "if_decision_score": -0.09
}

def evaluate_profile(name, profile):
    anom_sc = scale_anomaly_score(profile["if_decision_score"])
    ind_sc = compute_indicator_score(profile)
    final_sc = np.clip(0.60 * anom_sc + 0.40 * ind_sc, 0, 100)
    return {
        "profile": name, 
        "anomaly_score": round(anom_sc, 2), 
        "indicator_score": round(ind_sc, 2), 
        "risk_score": round(final_sc, 2), 
        "risk_level": assign_risk_level(final_sc)
    }

synth_results = pd.DataFrame([
    evaluate_profile("Normal Product", synthetic_normal),
    evaluate_profile("Moderate Suspicious", synthetic_moderate),
    evaluate_profile("Critical Suspicious", synthetic_critical)
])

print("\n" + "=" * 80)
print("SYNTHETIC PROFILE MONOTONICITY VALIDATION")
print("=" * 80)
display(synth_results)

# 3. Freeze Final Calibration Configuration
calibration_config = {
    "version": "phase5a2_frozen",
    "bounds": {"low_q01": float(low_q), "high_q99": float(high_q)},
    "risk_weights": {"isolation_forest": 0.60, "risk_indicators": 0.40},
    "risk_thresholds": {"LOW": [0, 25], "MEDIUM": [25, 50], "HIGH": [50, 75], "CRITICAL": [75, 100]},
    "indicator_weights": indicator_weights
}

with open(CALIBRATION_FILE, "w", encoding="utf-8") as f:
    json.dump(calibration_config, f, indent=4)

print("\n" + "=" * 80)
print("REFINED RISK LEVEL BREAKDOWN")
print("=" * 80)
print(df["risk_level"].value_counts().to_string())
print(f"\n✓ Calibration config frozen at: {CALIBRATION_FILE}")

PHASE 5A.2 — INDICATOR PREVALENCE AUDIT (% of 10,000 Products Triggered)
extreme_rating           :  15.00%
very_low_review_count    :  25.72%
high_rating_low_reviews  :  14.89%
missing_seller           :   0.62%
very_short_title         :   0.14%
very_short_description   :  43.36%

SYNTHETIC PROFILE MONOTONICITY VALIDATION


,profile,anomaly_score,indicator_score,risk_score,risk_level
0,Normal Product,30.55,0.00,18.33,LOW
1,Moderate Suspicious,73.94,81.82,77.09,CRITICAL
2,Critical Suspicious,100.00,100.00,100.00,CRITICAL



REFINED RISK LEVEL BREAKDOWN
risk_level
LOW         5096
MEDIUM      3244
HIGH        1502
CRITICAL     158

✓ Calibration config frozen at: C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\models\trustguard_risk_calibration.json


In [28]:
# ============================================================
# PHASE 5B.1 — FEATURE PARITY TEST (FIXED RE-EVALUATION)
# ============================================================
import os
import sys
import pandas as pd
import numpy as np

BASE_DIR = r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml"
sys.path.append(os.path.join(BASE_DIR, "ml-service"))

import risk_engine

FEATURE_DATASET = os.path.join(BASE_DIR, "data", "raw", "amazon_metadata_26_features.csv.gz")
df_samples = pd.read_csv(FEATURE_DATASET, nrows=10)

print("=" * 80)
print("RUNNING DIRECT MODEL INFERENCE TEST (10 Sample Records)")
print("=" * 80)

for idx, row in df_samples.iterrows():
    # Extract only the 22 features required by the trained model
    feature_row = {col: row[col] for col in risk_engine.MODEL_FEATURES if col in row}
    
    # Run through the core inference components of risk_engine directly
    feature_df = pd.DataFrame([feature_row])
    X_imputed = risk_engine.IMPUTER.transform(feature_df[risk_engine.MODEL_FEATURES])
    
    prediction = int(risk_engine.MODEL.predict(X_imputed)[0])
    decision_score = float(risk_engine.MODEL.decision_function(X_imputed)[0])
    
    anomaly_score = risk_engine.calculate_anomaly_score(decision_score)
    indicator_score = risk_engine.calculate_indicator_score(feature_row)
    
    risk_score = float(np.clip(
        risk_engine.ISOLATION_WEIGHT * anomaly_score + 
        risk_engine.INDICATOR_WEIGHT * indicator_score, 0, 100
    ))
    risk_level = risk_engine.assign_risk_level(risk_score)
    
    asin = row.get("parent_asin", row.get("asin", f"ITEM_{idx+1}"))
    print(f"Product #{idx+1} [ASIN: {asin}]:")
    print(f"  Risk Score : {round(risk_score, 2)} ({risk_level})")
    print(f"  Anomaly Sc : {round(anomaly_score, 2)}")
    print(f"  IF Pred    : {prediction}")

print("\n" + "=" * 80)
print("✓ Direct Parity Test Complete!")
print("=" * 80)

RUNNING DIRECT MODEL INFERENCE TEST (10 Sample Records)
Product #1 [ASIN: B00MCW7G9M]:
  Risk Score : 37.41 (MEDIUM)
  Anomaly Sc : 62.35
  IF Pred    : 1
Product #2 [ASIN: B00YT6XQSE]:
  Risk Score : 67.45 (HIGH)
  Anomaly Sc : 63.93
  IF Pred    : 1
Product #3 [ASIN: B07SM135LS]:
  Risk Score : 12.12 (LOW)
  Anomaly Sc : 14.13
  IF Pred    : 1
Product #4 [ASIN: B089CNGZCW]:
  Risk Score : 25.21 (MEDIUM)
  Anomaly Sc : 35.96
  IF Pred    : 1
Product #5 [ASIN: B004E2Z88O]:
  Risk Score : 13.5 (LOW)
  Anomaly Sc : 22.51
  IF Pred    : 1
Product #6 [ASIN: B00TX536EK]:
  Risk Score : 6.38 (LOW)
  Anomaly Sc : 10.63
  IF Pred    : 1
Product #7 [ASIN: B07BJ7ZZL7]:
  Risk Score : 13.91 (LOW)
  Anomaly Sc : 23.18
  IF Pred    : 1
Product #8 [ASIN: B005G99O2U]:
  Risk Score : 16.12 (LOW)
  Anomaly Sc : 26.86
  IF Pred    : 1
Product #9 [ASIN: B01MCZP7RF]:
  Risk Score : 50.27 (HIGH)
  Anomaly Sc : 35.29
  IF Pred    : 1
Product #10 [ASIN: B00R6R82HS]:
  Risk Score : 11.59 (LOW)
  Anomaly Sc : 

In [29]:
# ============================================================
# PHASE 5B.2 — RAW METADATA → PRODUCTION FEATURE PARITY
# Cell 1 — Configuration & Imports
# ============================================================

import os
import sys
import json
import numpy as np
import pandas as pd

BASE_DIR = r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml"

ML_SERVICE_DIR = os.path.join(BASE_DIR, "ml-service")
FEATURE_DATASET = os.path.join(
    BASE_DIR,
    "data",
    "raw",
    "amazon_metadata_26_features.csv.gz"
)

PARITY_REPORT = os.path.join(
    BASE_DIR,
    "reports",
    "phase5b2_raw_feature_parity_report.json"
)

sys.path.insert(0, ML_SERVICE_DIR)

import risk_engine

print("=" * 80)
print("PHASE 5B.2 — RAW METADATA → PRODUCTION FEATURE PARITY")
print("=" * 80)

print("risk_engine loaded from:")
print(risk_engine.__file__)

print("\nModel features:")
for i, feature in enumerate(risk_engine.MODEL_FEATURES):
    print(f"{i:02d} -> {feature}")

PHASE 5B.2 — RAW METADATA → PRODUCTION FEATURE PARITY
risk_engine loaded from:
C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\ml-service\risk_engine.py

Model features:
00 -> title_length
01 -> title_word_count
02 -> uppercase_ratio
03 -> special_character_ratio
04 -> description_length
05 -> description_word_count
06 -> feature_count
07 -> feature_text_length
08 -> category_count
09 -> image_count
10 -> video_count
11 -> has_videos
12 -> seller_missing
13 -> seller_name_length
14 -> price_missing
15 -> average_rating
16 -> rating_number
17 -> log_rating_number
18 -> rating_extremeness
19 -> high_rating
20 -> low_review_count
21 -> high_rating_low_reviews


In [30]:
# ============================================================
# Cell 2 — Load Reference Feature Dataset
# ============================================================

df = pd.read_csv(FEATURE_DATASET, nrows=10)

print("=" * 80)
print("REFERENCE DATASET")
print("=" * 80)

print("Shape:", df.shape)

print("\nColumns:")
for i, col in enumerate(df.columns):
    print(f"{i:02d} -> {col}")

print("\nFirst 5 parent ASINs:")
print(df["parent_asin"].head().tolist())

REFERENCE DATASET
Shape: (10, 27)

Columns:
00 -> parent_asin
01 -> title_length
02 -> title_word_count
03 -> uppercase_ratio
04 -> special_character_ratio
05 -> description_length
06 -> description_word_count
07 -> feature_count
08 -> feature_text_length
09 -> category_count
10 -> image_count
11 -> video_count
12 -> has_videos
13 -> seller_missing
14 -> seller_name_length
15 -> price_numeric
16 -> price_missing
17 -> price_ratio_to_category
18 -> log_price_ratio
19 -> price_anomaly
20 -> average_rating
21 -> rating_number
22 -> log_rating_number
23 -> rating_extremeness
24 -> high_rating
25 -> low_review_count
26 -> high_rating_low_reviews

First 5 parent ASINs:
['B00MCW7G9M', 'B00YT6XQSE', 'B07SM135LS', 'B089CNGZCW', 'B004E2Z88O']


In [31]:
# ============================================================
# Cell 3 — RAW METADATA FIELD INSPECTION
# ============================================================

print("=" * 80)
print("AVAILABLE DATASET FIELDS")
print("=" * 80)

for col in df.columns:
    print(f"{col:<35} -> {type(df.iloc[0][col]).__name__}")

print("\n" + "=" * 80)
print("REFERENCE FEATURE VALUES — FIRST PRODUCT")
print("=" * 80)

for feature in risk_engine.MODEL_FEATURES:
    print(f"{feature:<30} : {df.iloc[0][feature]}")

AVAILABLE DATASET FIELDS
parent_asin                         -> str
title_length                        -> int64
title_word_count                    -> int64
uppercase_ratio                     -> float64
special_character_ratio             -> float64
description_length                  -> int64
description_word_count              -> int64
feature_count                       -> int64
feature_text_length                 -> int64
category_count                      -> int64
image_count                         -> int64
video_count                         -> int64
has_videos                          -> int64
seller_missing                      -> int64
seller_name_length                  -> int64
price_numeric                       -> float64
price_missing                       -> int64
price_ratio_to_category             -> float64
log_price_ratio                     -> float64
price_anomaly                       -> float64
average_rating                      -> float64
rating_number     

In [32]:
# ============================================================
# Cell 4 — INSPECT RISK ENGINE FEATURE EXTRACTION INTERFACE
# ============================================================

print("=" * 80)
print("RISK ENGINE PUBLIC FUNCTIONS / CONSTANTS")
print("=" * 80)

public_items = [
    name for name in dir(risk_engine)
    if not name.startswith("_")
]

for name in public_items:
    obj = getattr(risk_engine, name)

    if callable(obj):
        print(f"FUNCTION : {name}")
    elif name.isupper():
        print(f"CONSTANT : {name}")

print("\n" + "=" * 80)
print("POTENTIAL FEATURE EXTRACTION FUNCTIONS")
print("=" * 80)

keywords = [
    "feature",
    "extract",
    "metadata",
    "product",
    "predict"
]

for name in public_items:
    if any(k in name.lower() for k in keywords):
        obj = getattr(risk_engine, name)

        if callable(obj):
            print(f"\n{name}")
            print("-" * 60)

            try:
                import inspect
                print(inspect.signature(obj))
                print(inspect.getsource(obj)[:3000])
            except Exception as e:
                print("Could not inspect source:", e)

RISK ENGINE PUBLIC FUNCTIONS / CONSTANTS
CONSTANT : ARTIFACT
CONSTANT : BASE_DIR
CONSTANT : CALIBRATION
CONSTANT : CALIBRATION_FILE
CONSTANT : EXPLANATION_TEXT
CONSTANT : HIGH_Q99
CONSTANT : IMPUTER
CONSTANT : INDICATOR_WEIGHT
CONSTANT : INDICATOR_WEIGHTS
CONSTANT : ISOLATION_WEIGHT
CONSTANT : LOW_Q01
CONSTANT : MODEL
CONSTANT : MODEL_FEATURES
CONSTANT : MODEL_FILE
FUNCTION : assign_risk_level
FUNCTION : calculate_anomaly_score
FUNCTION : calculate_indicator_score
FUNCTION : extract_features
FUNCTION : extract_indicators
FUNCTION : generate_explanations
FUNCTION : parse_array
FUNCTION : parse_categories
FUNCTION : parse_features
FUNCTION : parse_images
FUNCTION : parse_price
FUNCTION : parse_videos
FUNCTION : predict_product_risk
FUNCTION : safe_number
FUNCTION : safe_text
FUNCTION : special_character_ratio
FUNCTION : uppercase_ratio
FUNCTION : word_count

POTENTIAL FEATURE EXTRACTION FUNCTIONS

extract_features
------------------------------------------------------------
(product)
def

In [33]:
# ============================================================
# PHASE 5B.2 — Cell 5
# Construct Raw Product Payloads From Reference Features
# ============================================================

def build_raw_product(row):
    """
    Construct a raw metadata payload whose engineered
    feature values are known from the Phase 4B.2 dataset.

    This lets us test the production extraction path:
        raw metadata -> extract_features()
    """

    feature_count = int(row["feature_count"])
    feature_text_length = int(row["feature_text_length"])

    category_count = int(row["category_count"])
    image_count = int(row["image_count"])
    video_count = int(row["video_count"])

    # --------------------------------------------------------
    # Construct feature strings.
    #
    # We need:
    #   len(features) == feature_count
    #   sum(len(feature)) == feature_text_length
    # --------------------------------------------------------

    raw_features = []

    if feature_count > 0:

        remaining = feature_text_length

        for i in range(feature_count):

            if i == feature_count - 1:
                length = max(remaining, 0)
            else:
                # Keep enough characters for remaining fields
                length = max(
                    1,
                    remaining // (feature_count - i)
                )

            raw_features.append("x" * length)

            remaining -= length

    # --------------------------------------------------------
    # Categories
    # --------------------------------------------------------

    raw_categories = [
        f"Category {i}"
        for i in range(category_count)
    ]

    # --------------------------------------------------------
    # Images
    # --------------------------------------------------------

    raw_images = {
        "large": [
            f"https://example.com/image_{i}.jpg"
            for i in range(image_count)
        ]
    }

    # --------------------------------------------------------
    # Videos
    # --------------------------------------------------------

    raw_videos = {
        "title": [
            f"Video {i}"
            for i in range(video_count)
        ],
        "url": [
            f"https://example.com/video_{i}"
            for i in range(video_count)
        ],
        "user_id": [
            f"user_{i}"
            for i in range(video_count)
        ]
    }

    # --------------------------------------------------------
    # Seller
    # --------------------------------------------------------

    seller = (
        None
        if int(row["seller_missing"]) == 1
        else "Test Seller"
    )

    # --------------------------------------------------------
    # Rating
    # --------------------------------------------------------

    average_rating = float(row["average_rating"])

    rating_number = float(row["rating_number"])

    # --------------------------------------------------------
    # Title reconstruction
    #
    # Exact title text cannot be reconstructed from length /
    # ratios alone. Therefore we preserve the exact length,
    # but separately audit these fields below.
    # --------------------------------------------------------

    title_length = int(row["title_length"])

    title = "A" * title_length

    # --------------------------------------------------------
    # Description
    # --------------------------------------------------------

    description_length = int(row["description_length"])

    description = "x" * description_length

    # --------------------------------------------------------
    # Price is deliberately irrelevant to the 22-feature model.
    # --------------------------------------------------------

    return {
        "parent_asin": row.get("parent_asin"),

        "title": title,
        "description": description,

        "features": raw_features,
        "categories": raw_categories,
        "images": raw_images,
        "videos": raw_videos,

        "seller": seller,

        "average_rating": average_rating,
        "rating_number": rating_number
    }


raw_products = [
    build_raw_product(row)
    for _, row in df.iterrows()
]

print("=" * 80)
print("RAW PRODUCT PAYLOAD CONSTRUCTION")
print("=" * 80)

print("Products constructed:", len(raw_products))

print("\nExample payload:")
print(json.dumps(raw_products[0], indent=2)[:3000])

RAW PRODUCT PAYLOAD CONSTRUCTION
Products constructed: 10

Example payload:
{
  "parent_asin": "B00MCW7G9M",
  "title": "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA",
  "description": "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
  "features": [],
  "categories": [
    "Category 0",
    "Category 1",
    "Category 2"
  ],
  "images": {
    "large": [
      "https://example.com/image_0.j

In [34]:
# ============================================================
# PHASE 5B.2 — Cell 6
# RAW → ENGINEERED FEATURE EXTRACTION TEST
# ============================================================

print("=" * 80)
print("RAW → PRODUCTION FEATURE EXTRACTION")
print("=" * 80)

extracted_rows = []

for product in raw_products:

    extracted = risk_engine.extract_features(product)

    extracted_rows.append(extracted)

production_features = pd.DataFrame(extracted_rows)

print("Production feature matrix:")
print(production_features.shape)

print("\nExpected:")
print((len(raw_products), len(risk_engine.MODEL_FEATURES)))

print("\nFeature columns:")
for i, col in enumerate(production_features.columns):
    print(f"{i:02d} -> {col}")

RAW → PRODUCTION FEATURE EXTRACTION
Production feature matrix:
(10, 22)

Expected:
(10, 22)

Feature columns:
00 -> title_length
01 -> title_word_count
02 -> uppercase_ratio
03 -> special_character_ratio
04 -> description_length
05 -> description_word_count
06 -> feature_count
07 -> feature_text_length
08 -> category_count
09 -> image_count
10 -> video_count
11 -> has_videos
12 -> seller_missing
13 -> seller_name_length
14 -> price_missing
15 -> average_rating
16 -> rating_number
17 -> log_rating_number
18 -> rating_extremeness
19 -> high_rating
20 -> low_review_count
21 -> high_rating_low_reviews


In [35]:
# ============================================================
# PHASE 5B.2 — Cell 7
# EXACT FEATURE PARITY COMPARISON
# ============================================================

print("=" * 80)
print("EXACT FEATURE PARITY COMPARISON")
print("=" * 80)

reference = df[
    risk_engine.MODEL_FEATURES
].copy()

production = production_features[
    risk_engine.MODEL_FEATURES
].copy()

comparison_rows = []

for feature in risk_engine.MODEL_FEATURES:

    ref_values = pd.to_numeric(
        reference[feature],
        errors="coerce"
    )

    prod_values = pd.to_numeric(
        production[feature],
        errors="coerce"
    )

    differences = (
        prod_values - ref_values
    ).abs()

    comparison_rows.append({
        "feature": feature,
        "reference_mean": ref_values.mean(),
        "production_mean": prod_values.mean(),
        "max_abs_difference": differences.max(),
        "mean_abs_difference": differences.mean(),
        "exact_matches": int(
            (differences == 0).sum()
        ),
        "mismatches": int(
            (differences != 0).sum()
        )
    })

feature_report = pd.DataFrame(comparison_rows)

print(
    feature_report[
        [
            "feature",
            "max_abs_difference",
            "mean_abs_difference",
            "exact_matches",
            "mismatches"
        ]
    ].to_string(index=False)
)

total_mismatches = int(
    feature_report["mismatches"].sum()
)

print("\n" + "=" * 80)
print("PARITY SUMMARY")
print("=" * 80)

print("Total feature comparisons :", len(df) * len(risk_engine.MODEL_FEATURES))
print("Total mismatches          :", total_mismatches)

if total_mismatches == 0:
    print("\n✓ EXACT FEATURE PARITY")
else:
    print("\n⚠ FEATURE MISMATCHES DETECTED")

EXACT FEATURE PARITY COMPARISON
                feature  max_abs_difference  mean_abs_difference  exact_matches  mismatches
           title_length        0.000000e+00         0.000000e+00             10           0
       title_word_count        3.000000e+01         1.450000e+01              0          10
        uppercase_ratio        8.936170e-01         7.046231e-01              1           9
special_character_ratio        6.896552e-02         3.513407e-02              2           8
     description_length        0.000000e+00         0.000000e+00             10           0
 description_word_count        1.040000e+02         4.290000e+01              3           7
          feature_count        0.000000e+00         0.000000e+00             10           0
    feature_text_length        0.000000e+00         0.000000e+00             10           0
         category_count        0.000000e+00         0.000000e+00             10           0
            image_count        0.000000e+00     

In [36]:
# ============================================================
# PHASE 5B.2 — Cell 8
# FEATURE MISMATCH DIAGNOSTICS
# ============================================================

mismatch_features = feature_report[
    feature_report["mismatches"] > 0
]["feature"].tolist()

print("=" * 80)
print("FEATURE MISMATCH DIAGNOSTICS")
print("=" * 80)

if not mismatch_features:

    print("✓ No feature mismatches.")

else:

    for feature in mismatch_features:

        print("\n" + "-" * 70)
        print("FEATURE:", feature)

        ref = reference[feature]
        prod = production[feature]

        diff = (prod - ref).abs()

        bad_indices = diff[
            diff > 0
        ].index.tolist()

        print("Mismatch count:", len(bad_indices))

        for idx in bad_indices[:5]:

            print(
                f"Row {idx}: "
                f"reference={ref.iloc[idx]} | "
                f"production={prod.iloc[idx]} | "
                f"difference={diff.iloc[idx]}"
            )

FEATURE MISMATCH DIAGNOSTICS

----------------------------------------------------------------------
FEATURE: title_word_count
Mismatch count: 10
Row 0: reference=6 | production=1 | difference=5
Row 1: reference=6 | production=1 | difference=5
Row 2: reference=31 | production=1 | difference=30
Row 3: reference=28 | production=1 | difference=27
Row 4: reference=6 | production=1 | difference=5

----------------------------------------------------------------------
FEATURE: uppercase_ratio
Mismatch count: 9
Row 1: reference=0.5 | production=1.0 | difference=0.5
Row 2: reference=0.1946308724832214 | production=1.0 | difference=0.8053691275167786
Row 3: reference=0.1428571428571428 | production=1.0 | difference=0.8571428571428572
Row 4: reference=0.1818181818181818 | production=1.0 | difference=0.8181818181818182
Row 5: reference=0.1956521739130435 | production=1.0 | difference=0.8043478260869565

----------------------------------------------------------------------
FEATURE: special_charac

In [37]:
# ============================================================
# PHASE 5B.2 — Cell 9
# END-TO-END PRODUCTION PREDICTION PARITY
# ============================================================

print("=" * 80)
print("END-TO-END PRODUCTION PREDICTION TEST")
print("=" * 80)

results = []

for idx, product in enumerate(raw_products):

    result = risk_engine.predict_product_risk(product)

    results.append({
        "index": idx,
        "parent_asin": product.get("parent_asin"),
        "risk_score": result["risk_score"],
        "risk_level": result["risk_level"],
        "is_anomaly": result["is_anomaly"],
        "anomaly_score": result["anomaly_score"],
        "indicator_score": result["indicator_score"],
        "if_prediction": result["isolation_forest"]["prediction"],
        "decision_score": result["isolation_forest"]["decision_score"]
    })

prediction_results = pd.DataFrame(results)

display(prediction_results)

END-TO-END PRODUCTION PREDICTION TEST


,index,parent_asin,risk_score,risk_level,is_anomaly,anomaly_score,indicator_score,if_prediction,decision_score
0,0,B00MCW7G9M,40.79,MEDIUM,False,67.98,0.00,1,0.019610
1,1,B00YT6XQSE,80.29,CRITICAL,True,85.33,72.73,-1,-0.008385
2,2,B07SM135LS,50.12,HIGH,False,77.47,9.09,1,0.004304
3,3,B089CNGZCW,53.51,HIGH,True,83.12,9.09,-1,-0.004806
4,4,B004E2Z88O,30.20,MEDIUM,False,50.34,0.00,1,0.048077
5,5,B00TX536EK,31.10,MEDIUM,False,51.83,0.00,1,0.045679
6,6,B07BJ7ZZL7,41.53,MEDIUM,False,69.22,0.00,1,0.017618
7,7,B005G99O2U,30.80,MEDIUM,False,51.33,0.00,1,0.046478
8,8,B01MCZP7RF,77.62,CRITICAL,True,80.88,72.73,-1,-0.001202
9,9,B00R6R82HS,37.65,MEDIUM,False,62.74,0.00,1,0.028064


In [38]:
# ============================================================
# PHASE 5B.2 — Cell 10
# PRODUCTION VS NOTEBOOK PREDICTION PARITY
# ============================================================

print("=" * 80)
print("PRODUCTION VS NOTEBOOK PREDICTION PARITY")
print("=" * 80)

reference_predictions = pd.read_csv(
    os.path.join(
        BASE_DIR,
        "data",
        "raw",
        "phase5a_validation_results_corrected.csv.gz"
    ),
    nrows=len(df)
)

comparison = pd.DataFrame({
    "parent_asin": reference_predictions["parent_asin"],

    "reference_prediction":
        reference_predictions["if_prediction"],

    "production_prediction":
        prediction_results["if_prediction"],

    "reference_decision":
        reference_predictions["if_decision_score"],

    "production_decision":
        prediction_results["decision_score"]
})

comparison["prediction_match"] = (
    comparison["reference_prediction"]
    ==
    comparison["production_prediction"]
)

comparison["decision_difference"] = (
    comparison["reference_decision"]
    -
    comparison["production_decision"]
).abs()

print(
    comparison[
        [
            "parent_asin",
            "reference_prediction",
            "production_prediction",
            "prediction_match",
            "decision_difference"
        ]
    ].to_string(index=False)
)

prediction_disagreements = int(
    (~comparison["prediction_match"]).sum()
)

max_decision_difference = float(
    comparison["decision_difference"].max()
)

print("\n" + "=" * 80)
print("FINAL MODEL PARITY")
print("=" * 80)

print(
    "Prediction disagreements :",
    prediction_disagreements
)

print(
    "Maximum decision difference :",
    max_decision_difference
)

if prediction_disagreements == 0:
    print("\n✓ MODEL PREDICTIONS MATCH")
else:
    print("\n⚠ MODEL PREDICTION MISMATCH")

PRODUCTION VS NOTEBOOK PREDICTION PARITY
parent_asin  reference_prediction  production_prediction  prediction_match  decision_difference
 B00MCW7G9M                     1                      1              True             0.009092
 B00YT6XQSE                     1                     -1             False             0.034534
 B07SM135LS                     1                      1              True             0.102183
 B089CNGZCW                     1                     -1             False             0.076088
 B004E2Z88O                     1                      1              True             0.044902
 B00TX536EK                     1                      1              True             0.066470
 B07BJ7ZZL7                     1                      1              True             0.074276
 B005G99O2U                     1                      1              True             0.039476
 B01MCZP7RF                     1                     -1             False             0.073552

In [39]:
# ============================================================
# PHASE 5B.2 — Cell 11
# SAVE PARITY REPORT
# ============================================================

feature_mismatch_count = int(
    feature_report["mismatches"].sum()
)

report = {
    "phase": "5B.2",

    "dataset_rows": int(len(df)),

    "model_features": int(
        len(risk_engine.MODEL_FEATURES)
    ),

    "feature_parity": {
        "total_comparisons": int(
            len(df) * len(risk_engine.MODEL_FEATURES)
        ),
        "mismatches": feature_mismatch_count,
        "passed": feature_mismatch_count == 0
    },

    "prediction_parity": {
        "rows_tested": int(len(comparison)),
        "prediction_disagreements":
            prediction_disagreements,
        "max_decision_difference":
            max_decision_difference,
        "passed":
            prediction_disagreements == 0
    },

    "production_model": {
        "contamination":
            float(risk_engine.MODEL.contamination),

        "n_estimators":
            int(risk_engine.MODEL.n_estimators),

        "n_features_in":
            int(risk_engine.MODEL.n_features_in_)
    },

    "calibration": {
        "low_q01":
            float(risk_engine.LOW_Q01),

        "high_q99":
            float(risk_engine.HIGH_Q99),

        "isolation_weight":
            float(risk_engine.ISOLATION_WEIGHT),

        "indicator_weight":
            float(risk_engine.INDICATOR_WEIGHT)
    }
}

with open(
    PARITY_REPORT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=4
    )

print("=" * 80)
print("PHASE 5B.2 COMPLETE")
print("=" * 80)

print(
    "Feature parity passed :",
    report["feature_parity"]["passed"]
)

print(
    "Prediction parity passed :",
    report["prediction_parity"]["passed"]
)

print("\nReport saved:")
print(PARITY_REPORT)

PHASE 5B.2 COMPLETE
Feature parity passed : False
Prediction parity passed : False

Report saved:
C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\reports\phase5b2_raw_feature_parity_report.json
